# 🪐 Quantum Interplanetary Traffic Network
### จำลองระบบเครือข่ายควอนตัมระหว่างดาวเคราะห์ — Google Colab (ภาษาไทย)
> แปลงมาจาก Three.js 3D Visualization — ไม่มีกราฟิก ใช้ Logic การจำลอง + Console Log เท่านั้น

---
**Group Assignment 1 · New Network Group**
`673380302-8 Yoko · 673380510-1 Eiangey · 673380050-9 Pok · 673308270-5 Jump · 673380517-7 Mark`


## 1 · นำเข้า Library และเตรียมระบบ

ใช้แค่ Standard Library ของ Python — ไม่ต้องติดตั้งเพิ่ม

| Library | ใช้ทำอะไร |
|---------|-----------|
| `math` | คำนวณตำแหน่งวงโคจร (cos, sin, hypot) |
| `random` | สุ่มทิศทางส่งข้อมูลและความเร็วจุด |
| `time` | แสดง timestamp ใน log |


In [ ]:
import random
import math
import time


## 2 · ข้อมูลดาวเคราะห์ (QTN Node)

แต่ละดาวเคราะห์คือ **โหนดเครือข่าย QTN (Quantum Traffic Network)**

| คีย์ | ความหมาย |
|------|----------|
| `orbit` | รัศมีวงโคจร (หน่วย AU) — ยิ่งมาก ยิ่งอยู่ห่างดวงอาทิตย์ |
| `spd` | ความเร็วเชิงมุม — ดาวพุธเร็วสุด (4.7), ดาวเนปจูนช้าสุด (0.54) |
| `radius` | ขนาดดาว (เก็บไว้อ้างอิง ไม่ได้ใช้ในการคำนวณ) |
| `info` | ข้อมูล QTN node ของดาวนั้น |

> **บรรทัดสำคัญ:** `"orbit": 44, "spd": 2.9` หมายความว่า โลกหมุนรอบวงกลมรัศมี 44 ด้วยความเร็ว 2.9 เท่า


In [ ]:
PLANET_DATA = [
    {"name": "Mercury", "color": "gray",   "radius": 1.8, "orbit": 22,  "spd": 4.7,
     "info": "QTN-α  · ศูนย์กลาง Mercury\nคู่ Entangle: 320\nLatency: 0 ms (quantum)"},
    {"name": "Venus",   "color": "orange", "radius": 3.1, "orbit": 32,  "spd": 3.5,
     "info": "QTN-β  · สถานีถ่ายทอด Venus\nคู่ Entangle: 840\nLatency: 0 ms (quantum)"},
    {"name": "Earth",   "color": "blue",   "radius": 3.9, "orbit": 44,  "spd": 2.9,
     "info": "QTN-PRIME · โหนดต้นทาง (Earth)\nUniversal Traffic Brain\nAI nodes: 12,400"},
    {"name": "Mars",    "color": "red",    "radius": 2.8, "orbit": 58,  "spd": 2.4,
     "info": "QTN-δ  · อาณานิคม Mars\nAI traffic nodes: 4,200\nLatency: 0 ms (quantum)"},
    {"name": "Jupiter", "color": "tan",    "radius": 7.2, "orbit": 82,  "spd": 1.3,
     "info": "QTN-ε  · Quantum Repeater\nBandwidth: ∞ (entangled)\nMycelium layer active"},
    {"name": "Saturn",  "color": "wheat",  "radius": 5.8, "orbit": 108, "spd": 0.97,
     "info": "QTN-ζ  · DNA Storage node\n1 Exabyte ต่อ node\nRing relay array: online"},
    {"name": "Uranus",  "color": "cyan",   "radius": 4.4, "orbit": 132, "spd": 0.68,
     "info": "QTN-η  · Mycelium layer\nเครือข่ายซ่อมตัวเอง\nBio-quantum bridge: active"},
    {"name": "Neptune", "color": "navy",   "radius": 4.1, "orbit": 155, "spd": 0.54,
     "info": "QTN-θ  · Edge node\nจุดสิ้นสุดเครือข่าย\nForward error correction: on"},
]

# คู่เชื่อมต่อด้วย Quantum Beam (ลิงก์ entangled ถาวร)
# เช่น (2,3) = Earth ↔ Mars เชื่อมกันตลอดเวลา
BEAM_PAIRS = [(2,3),(0,2),(1,2),(3,4),(4,5),(5,6),(6,7),(2,4),(1,3)]

NUM_PLANETS = len(PLANET_DATA)
print(f"โหลดสำเร็จ: {NUM_PLANETS} QTN nodes")
print()
print("คู่ Quantum Beam (เชื่อมต่อถาวร):")
for a, b in BEAM_PAIRS:
    print(f"  {PLANET_DATA[a]['name']} ↔ {PLANET_DATA[b]['name']}")


## 3 · สถานะการจำลอง (Global State)

ตัวแปรเหล่านี้เปลี่ยนทุก tick — ตรงกับตัวแปร JS ในไฟล์ต้นฉบับ

| ตัวแปร | ตรงกับ JS | หน้าที่ |
|--------|-----------|---------|
| `angles` | `angles[]` | มุมวงโคจรปัจจุบัน (เรเดียน) ของแต่ละดาว |
| `positions` | `mesh.position` | ตำแหน่ง (x, z) คำนวณใหม่ทุก tick |
| `streams` | `streams[]` | รายการ stream ข้อมูลที่กำลังส่งอยู่ |
| `total_sent` | `totalSent` | จำนวนการส่งทั้งหมดสะสม |
| `sim_speed` | slider ความเร็ว | ตัวคูณความเร็ว (1.0 = ปกติ) |


In [ ]:
# แต่ละดาวเริ่มต้นที่มุมสุ่ม → ไม่เรียงกันเป็นเส้นตรง
angles    = [random.uniform(0, math.pi * 2) for _ in range(NUM_PLANETS)]
positions = [(0.0, 0.0)] * NUM_PLANETS   # (x, z) — อัปเดตทุก tick

streams           = []   # stream ที่กำลัง active อยู่
total_sent        = 0    # นับจำนวนแพ็กเกตที่ส่งไปทั้งหมด
stream_id_counter = 0    # ID ไว้ trace ใน log

# ปุ่มควบคุม (เหมือน UI ใน HTML)
sim_speed = 1.0    # ตัวคูณความเร็ว — เหมือน slider Speed
auto_mode = True   # ส่งอัตโนมัติหรือไม่ — เหมือนปุ่ม Auto Broadcast


## 4 · กลศาสตร์วงโคจร — `update_positions()`

> **สูตรสำคัญ:**
> ```
> x = cos(มุม) × รัศมีวงโคจร
> z = sin(มุม) × รัศมีวงโคจร
> ```
> นี่คือสมการวงกลมพาราเมตริก — ใช้คำนวณตำแหน่งดาวทุก tick
>
> มุมเพิ่มขึ้นทุก tick โดย:
> ```
> Δมุม = delta × ความเร็วดาว × 0.011
> ```
> ค่าคงที่ `0.011` คัดลอกมาจาก JS ตรงๆ เพื่อให้ timing เหมือนกัน


In [ ]:
def update_positions(delta: float) -> None:
    """
    เลื่อนมุมวงโคจรของทุกดาว และคำนวณตำแหน่ง (x, z) ใหม่

    ★ บรรทัดสำคัญ:
        angles[i] += delta * PLANET_DATA[i]['spd'] * 0.011
        → นี่คือการก้าวเชิงมุม — 0.011 คือค่าคงที่จาก JS
        → ดาวพุธ (spd=4.7) เร็วกว่าดาวเนปจูน (spd=0.54) ประมาณ 8.7 เท่า

        positions[i] = (cos(angle)*r, sin(angle)*r)
        → สมการวงกลม — x และ z คำนวณจาก cos และ sin ของมุม
    """
    for i in range(NUM_PLANETS):
        # ★ สำคัญ: ความเร็วเชิงมุม = delta × speed × 0.011
        angles[i] += delta * PLANET_DATA[i]["spd"] * 0.011

        r = PLANET_DATA[i]["orbit"]
        # ★ สำคัญ: สมการวงกลมพาราเมตริก
        positions[i] = (math.cos(angles[i]) * r, math.sin(angles[i]) * r)

# ทดสอบ
update_positions(1.0)
print("ตำแหน่งดาวเคราะห์ปัจจุบัน:")
for i, p in enumerate(PLANET_DATA):
    x, z = positions[i]
    print(f"  {p['name']:8s}  มุม={angles[i]:.3f} rad  ตำแหน่ง=({x:7.2f}, {z:7.2f})")


## 5 · คำนวณระยะห่าง — `distance()`

ระยะทางแบบ Euclidean ระหว่างดาว 2 ดวง
ใช้แสดงใน log ว่าแพ็กเกตต้องเดินทางไกลแค่ไหน


In [ ]:
def distance(a: int, b: int) -> float:
    """
    คำนวณระยะห่างระหว่างดาว a กับ b บนระนาบ 2 มิติ

    ★ บรรทัดสำคัญ:
        math.hypot(bx-ax, bz-az)
        → เทียบเท่า sqrt((bx-ax)² + (bz-az)²) แต่เขียนสั้นกว่า
    """
    ax, az = positions[a]
    bx, bz = positions[b]
    # ★ สำคัญ: hypot คือ Euclidean distance แบบย่อ
    return math.hypot(bx - ax, bz - az)

# ทดสอบ
print("ตารางระยะห่างระหว่างดาว (AU-units):")
print(f"  {'':10s}", end="")
for p in PLANET_DATA:
    print(f"  {p['name'][:4]:4s}", end="")
print()
for i, pi in enumerate(PLANET_DATA):
    print(f"  {pi['name']:10s}", end="")
    for j in range(NUM_PLANETS):
        if i == j:
            print(f"  {'—':4s}", end="")
        else:
            print(f"  {distance(i,j):4.0f}", end="")
    print()


## 6 · เส้นทาง Bézier — `bezier_progress()`

จุดข้อมูลไม่ได้บินตรง — แต่โค้งตามเส้น **Quadratic Bézier**

```
           จุดกึ่งกลาง (ยกสูงขึ้น)
          /     \
        /         \
      p0 ─────────── p1
    (ต้นทาง)      (ปลายทาง)
```

> **สูตรสำคัญ (lerp-of-lerp):**
> ```
> left  = lerp(p0, mid, t)
> right = lerp(mid, p1, t)
> ตำแหน่ง = lerp(left, right, t)   ← Quadratic Bézier
> ```
> `t` วิ่งจาก 0.0 (ดาวต้นทาง) → 1.0 (ดาวปลายทาง)
>
> ค่า `0.28` คือความสูงโค้ง = 28% ของระยะทางตรง (มาจาก JS ต้นฉบับ)


In [ ]:
def bezier_progress(p0: tuple, p1: tuple, t: float) -> tuple:
    """
    คำนวณตำแหน่งบนเส้น Quadratic Bézier ที่พารามิเตอร์ t

    ★ บรรทัดสำคัญ:
        lift = distance * 0.28
        → ความสูงโค้ง = 28% ของระยะทางตรง (ค่าจาก JS)

        lerp-of-lerp ใน x, z
        → สูตร Quadratic Bézier มาตรฐาน
    """
    mx = (p0[0] + p1[0]) / 2
    mz = (p0[1] + p1[1]) / 2
    # ★ สำคัญ: ความสูงโค้งตามสัดส่วนระยะทาง (0.28 จาก JS)
    _lift = math.hypot(p1[0]-p0[0], p1[1]-p0[1]) * 0.28

    # ★ สำคัญ: Quadratic Bézier ด้วย lerp-of-lerp
    lx = p0[0] + (mx - p0[0]) * t
    lz = p0[1] + (mz - p0[1]) * t
    rx = mx + (p1[0] - mx) * t
    rz = mz + (p1[1] - mz) * t
    return (lx + (rx - lx) * t, lz + (rz - lz) * t)

# สาธิต: แสดงตำแหน่งแพ็กเกตตลอดเส้นทาง โลก → ดาวพฤหัสบดี
p0, p1 = positions[2], positions[4]
print("เส้นทาง Bézier  โลก (Earth) → ดาวพฤหัสบดี (Jupiter)")
print(f"  {'t':>5}   {'x':>8}   {'z':>8}   ความคืบหน้า")
print("  " + "-" * 48)
for i in range(11):
    t = i / 10
    x, z = bezier_progress(p0, p1, t)
    bar = "█" * int(t * 25)
    print(f"  {t:5.2f}   {x:8.2f}   {z:8.2f}   [{bar:<25s}]")


## 7 · สร้าง Stream — `make_stream()`

**Stream** คือ 1 เหตุการณ์ส่งข้อมูล ประกอบด้วย **14 จุด (dot)**

แต่ละจุดมี:
- `t` — ความคืบหน้าบนเส้น Bézier (เริ่มเป็นลบ → จุดออกไม่พร้อมกัน)
- `speed` — ความเร็วสุ่มเล็กน้อย → จุดกระจายตัวเป็นธรรมชาติ
- `arrived` — flag ป้องกันการ log ซ้ำ


In [ ]:
def make_stream(src_idx: int, tgt_idx: int, count: int = 14) -> dict:
    """
    สร้าง stream ข้อมูลใหม่ที่มี count จุด (dot)

    ★ บรรทัดสำคัญ:
        t = -(i / count) * 1.15
        → t ติดลบ = จุดยังไม่ออกเดินทาง
        → จุดที่ 0 ออกก่อน, จุดที่ 13 ออกทีหลังสุด
        → 1.15 คือ buffer ระยะห่าง 15% (จาก JS)

        speed = 0.009 + random.uniform(0, 0.006)
        → ความเร็วสุ่มเล็กน้อย → ทำให้จุดไม่เรียงชิดกัน
    """
    global stream_id_counter
    stream_id_counter += 1
    dots = []
    for i in range(count):
        dots.append({
            # ★ สำคัญ: เวลาเริ่มติดลบ → จุดออกสลับกัน (stagger)
            "t":       -(i / count) * 1.15,
            # ★ สำคัญ: ความเร็วสุ่มเล็กน้อย (จาก JS: 0.009 + rand*0.006)
            "speed":    0.009 + random.uniform(0, 0.006),
            "arrived":  False,
        })
    return {
        "id":    stream_id_counter,
        "src":   src_idx,
        "tgt":   tgt_idx,
        "dots":  dots,
        "alive": True,
    }


## 8 · ฟังก์ชัน Log


In [ ]:
_log_lines = 0

def log(msg: str) -> None:
    """
    แสดง log พร้อม timestamp
    แทนที่การอัปเดต HUD ใน HTML ทั้งหมด
    """
    global _log_lines
    ts = time.strftime("%H:%M:%S")
    print(f"  {ts}  {msg}")
    _log_lines += 1


## 9 · ส่งข้อมูลและ Auto-Broadcast

`send_data()` — ส่ง stream และบันทึก log
`auto_broadcast()` — ยิง stream สุ่มอัตโนมัติทุก 0.75 วินาที (จำลอง)
ตรงกับฟังก์ชัน `maybeAutoSend()` ใน JS ต้นฉบับ


In [ ]:
def send_data(src_idx: int, tgt_idx: int) -> None:
    """
    ส่ง stream ข้อมูล 14 จุดจาก src → tgt

    ★ บรรทัดสำคัญ:
        if src_idx == tgt_idx: return
        → ป้องกันส่งหาตัวเอง (เหมือน JS)
    """
    global total_sent
    # ★ สำคัญ: ป้องกัน self-transmission
    if src_idx == tgt_idx:
        return
    s = make_stream(src_idx, tgt_idx)
    streams.append(s)
    total_sent += 1
    dist = distance(src_idx, tgt_idx)
    log(f"[ส่ง    #{s['id']:04d}] "
        f"{PLANET_DATA[src_idx]['name']:7s} → {PLANET_DATA[tgt_idx]['name']:7s}  "
        f"ระยะ={dist:6.1f} AU  จุด=14  รวม={total_sent}")


def auto_broadcast(raw_delta: float, auto_timer_ref: list) -> None:
    """
    สะสมเวลา แล้วยิง broadcast สุ่มทุก 0.75 วินาที

    ★ บรรทัดสำคัญ:
        auto_timer_ref[0] += raw_delta
        → ใช้ list แทน float เพราะ Python ส่งค่า float แบบ copy
          list ทำให้ฟังก์ชันนี้แก้ไขตัวเลขได้จริง

        if len(streams) < 40:
        → จำกัด stream สูงสุด 40 ตัว (เหมือน JS)
    """
    if not auto_mode:
        return
    # ★ สำคัญ: list trick สำหรับ mutable reference
    auto_timer_ref[0] += raw_delta
    if auto_timer_ref[0] > 0.75:
        auto_timer_ref[0] = 0.0
        if len(streams) < 40:     # ★ สำคัญ: จำกัดจำนวน stream
            a = random.randrange(NUM_PLANETS)
            b = random.randrange(NUM_PLANETS)
            while b == a:         # รับประกันว่า src ≠ tgt
                b = random.randrange(NUM_PLANETS)
            send_data(a, b)


## 10 · เดินหน้า Stream — `tick_streams()`

หัวใจของการจำลอง — เลื่อนตำแหน่งจุดทุกตัวในทุก tick

> **สูตรสำคัญ:**
> ```
> d["t"] += delta × d["speed"] × 55
> ```
> ค่าคงที่ `55` คือ speed scalar จาก JS ต้นฉบับ
>
> เมื่อ `t` เข้าช่วง `[0.97, 1.03]` → ถือว่า **ถึงปลายทาง** แล้ว


In [ ]:
def tick_streams(delta: float) -> None:
    """
    เดินหน้า dot ทุกตัวใน stream ที่ยังมีชีวิต

    ★ บรรทัดสำคัญ:
        d['t'] += delta * d['speed'] * 55
        → สมการเคลื่อนที่หลัก — 55 คือ speed scalar จาก JS

        if 0.97 <= d['t'] <= 1.03 and not d['arrived']:
        → ช่วงการมาถึง กว้างกว่า 1.0 เล็กน้อย
          เพื่อให้ทุก dot ยิง event ได้แน่นอนแม้ delta ใหญ่

        if all(d['t'] > 1.25 for d in s['dots']):
        → stream ตายเมื่อ ทุก dot ผ่านปลายทางไปแล้ว
    """
    for s in streams:
        if not s["alive"]:
            continue

        src_pos  = positions[s["src"]]
        tgt_pos  = positions[s["tgt"]]
        src_name = PLANET_DATA[s["src"]]["name"]
        tgt_name = PLANET_DATA[s["tgt"]]["name"]

        for d in s["dots"]:
            # ★ สำคัญ: เดิน dot ตามเส้น Bézier
            d["t"] += delta * d["speed"] * 55

            # ★ สำคัญ: event มาถึง — ยิงครั้งเดียวต่อ dot
            if 0.97 <= d["t"] <= 1.03 and not d["arrived"]:
                d["arrived"] = True
                pos = bezier_progress(src_pos, tgt_pos, 1.0)
                log(f"[มาถึง  #{s['id']:04d}] dot → {tgt_name:7s}  "
                    f"ตำแหน่ง=({pos[0]:6.1f},{pos[1]:6.1f})  จาก {src_name}")

        # ★ สำคัญ: ตาย = ทุก dot เกิน 1.25
        if all(d["t"] > 1.25 for d in s["dots"]):
            s["alive"] = False
            log(f"[เสร็จ   #{s['id']:04d}] {src_name} → {tgt_name} สำเร็จ")


def purge_dead_streams() -> None:
    """ลบ stream ที่เสร็จแล้วออกจาก list (วนถอยหลัง — ปลอดภัยกว่า)"""
    for i in range(len(streams) - 1, -1, -1):
        if not streams[i]["alive"]:
            streams.pop(i)


## 11 · แสดงสถิติ — `print_stats()`


In [ ]:
def print_stats(tick: int) -> None:
    """แสดงสรุปสถานะ — แทนแผง HUD ใน HTML"""
    print()
    print(f"  ── TICK {tick:04d} ─────────────────────────────────────")
    print(f"  ส่งทั้งหมด     : {total_sent} แพ็กเกต")
    print(f"  Stream ที่ active: {len(streams)}")
    print("  ตำแหน่งดาวเคราะห์ (x, z):")
    for i, p in enumerate(PLANET_DATA):
        x, z = positions[i]
        print(f"    {p['name']:8s}  ({x:7.2f}, {z:7.2f})")
    print()


## 12 · Loop การจำลองหลัก — `main()`

ทุก tick ทำตามลำดับนี้:
1. 🌍 เลื่อนตำแหน่งดาวเคราะห์
2. 📡 ตรวจสอบ auto-broadcast
3. 🚀 เดินหน้า dot ใน stream / log การมาถึง
4. 🧹 ลบ stream ที่เสร็จแล้ว
5. 📊 แสดงสถิติ (ทุก N tick)

| Parameter | ค่าเริ่มต้น | ความหมาย |
|-----------|------------|----------|
| `max_ticks` | 200 | จำนวน tick ที่จะรัน |
| `dt` | 0.05 วิ | เวลาต่อ tick (≈ 50ms เหมือน JS frame) |
| `stats_every` | 50 | แสดงสถิติทุก N tick |


In [ ]:
def main(max_ticks: int = 200, dt: float = 0.05, stats_every: int = 50) -> None:
    """
    Loop การจำลองหลัก

    ★ บรรทัดสำคัญ:
        delta = raw_delta * sim_speed
        → sim_speed คูณเวลา — 2.0 = เร็วขึ้น 2 เท่า (เหมือน slider JS)

        auto_timer = [0.0]
        → ใช้ list เพราะ Python ไม่ส่ง float แบบ reference
          ทำให้ auto_broadcast() แก้ค่าได้จริง
    """
    print("=" * 60)
    print("  🪐 QUANTUM INTERPLANETARY TRAFFIC NETWORK")
    print("     จำลองระบบเครือข่ายควอนตัม — Python Edition")
    print("=" * 60)
    print()

    # ★ สำคัญ: list wrapper ให้ auto_broadcast() แก้ค่าได้
    auto_timer = [0.0]

    for tick in range(1, max_ticks + 1):
        raw_delta = dt
        # ★ สำคัญ: คูณ sim_speed = ปรับความเร็วการจำลอง
        delta = raw_delta * sim_speed

        update_positions(delta)              # 1. เลื่อนดาวเคราะห์
        auto_broadcast(raw_delta, auto_timer) # 2. ส่งอัตโนมัติ
        tick_streams(delta)                  # 3. เดิน dot, log
        purge_dead_streams()                 # 4. ลบ stream เก่า

        if tick % stats_every == 0:
            print_stats(tick)

    print("=" * 60)
    print(f"  จำลองครบ {max_ticks} ticks")
    print(f"  แพ็กเกตที่ส่งทั้งหมด : {total_sent}")
    print(f"  บรรทัด log ทั้งหมด   : {_log_lines}")
    print("=" * 60)


## 13 · ▶ รันการจำลอง!

ปรับ `max_ticks` และ `stats_every` ตามต้องการ


In [ ]:
# ── Reset ทุกอย่างก่อนรัน ──────────────────────────────────────────────
angles            = [random.uniform(0, math.pi * 2) for _ in range(NUM_PLANETS)]
positions         = [(0.0, 0.0)] * NUM_PLANETS
streams.clear()
total_sent        = 0
stream_id_counter = 0
_log_lines        = 0

# ── รัน ───────────────────────────────────────────────────────────────
main(max_ticks=150, dt=0.05, stats_every=50)


## 14 · 🔬 พิเศษ — ตรวจสอบเส้นทาง Bézier แบบละเอียด

รัน cell นี้หลังการจำลองเพื่อดูทุกตำแหน่งของแพ็กเกตตลอดเส้นทาง


In [ ]:
# แสดงเส้นทาง Bézier แบบละเอียด: โลก → ดาวเนปจูน
src_idx, tgt_idx = 2, 7
p0, p1 = positions[src_idx], positions[tgt_idx]
steps  = 20

print(f"  เส้นทาง Bézier: {PLANET_DATA[src_idx]['name']} → {PLANET_DATA[tgt_idx]['name']}")
print(f"  {'t':>5}   {'x':>8}   {'z':>8}   ความคืบหน้า")
print("  " + "-" * 50)
for i in range(steps + 1):
    t   = i / steps
    x, z = bezier_progress(p0, p1, t)
    bar = "▓" * int(t * 30)
    print(f"  {t:5.2f}   {x:8.2f}   {z:8.2f}   [{bar:<30s}]")
